In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

project_path = "/content/drive/MyDrive/movie_genre_project"
os.makedirs(project_path, exist_ok=True)

print("Project folder ready:", project_path)

Project folder ready: /content/drive/MyDrive/movie_genre_project


In [ ]:
os.listdir("/content/drive/MyDrive/movie_genre_project")

['train_data.txt', 'checkpoints', 'final_model']

In [ ]:
os.listdir("/content/drive/MyDrive/movie_genre_project")

['train_data.txt', 'checkpoints', 'final_model']

In [ ]:
import pandas as pd

data_path = "/content/drive/MyDrive/movie_genre_project/train_data.txt"

df = pd.read_csv(
    data_path,
    sep=" ::: ",
    engine="python",
    header=None
)

df.columns = ["id", "title", "genre", "plot"]

df.head()

,id,title,genre,plot
0,1,Oscar et la dame rose (2009),drama,Listening in to a conversation between his doc...
1,2,Cupid (1997),thriller,A brother and sister with a past incestuous re...
2,3,"Young, Wild and Wonderful (1980)",adult,As the bus empties the students for their fiel...
3,4,The Secret Sin (1915),drama,To help their unemployed father make ends meet...
4,5,The Unrecovered (2007),drama,The film's title refers not only to the un-rec...


In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["genre"])

num_labels = len(label_encoder.classes_)
print("Number of classes:", num_labels)

Number of classes: 27


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df["plot"],
    df["label"],
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Train size:", len(X_train))
print("Test size:", len(X_test))

Train size: 43371
Test size: 10843


In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": list(X_train),
    "label": list(y_train)
})

test_dataset = Dataset.from_dict({
    "text": list(X_test),
    "label": list(y_test)
})

train_dataset

Dataset({
    features: ['text', 'label'],
    num_rows: 43371
})

In [ ]:
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [ ]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128   # keep 128 to reduce training time
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/43371 [00:00<?, ? examples/s]

Map:   0%|          | 0/10843 [00:00<?, ? examples/s]

In [ ]:
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_labels
)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from transformers import TrainingArguments

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/movie_genre_project/checkpoints",
    num_train_epochs=1,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    do_eval=True,
    save_strategy="epoch",   # ✅ better
    save_total_limit=2,
    fp16=True
)

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(labels, predictions, average="macro")
    }

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics
)

In [ ]:
trainer.train()

Step,Training Loss
500,1.744542
1000,1.423565
1500,1.269082
2000,1.209483
2500,1.222890


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=2711, training_loss=1.3555309952778467, metrics={'train_runtime': 222.5887, 'train_samples_per_second': 194.848, 'train_steps_per_second': 12.179, 'total_flos': 1436951250918144.0, 'train_loss': 1.3555309952778467, 'epoch': 1.0})

In [ ]:
trainer.evaluate()

{'eval_loss': 1.1368796825408936,
 'eval_accuracy': 0.6536936272249377,
 'eval_macro_f1': 0.3793268562460494,
 'eval_runtime': 10.8736,
 'eval_samples_per_second': 997.186,
 'eval_steps_per_second': 62.353,
 'epoch': 1.0}

In [ ]:
trainer.save_model("/content/drive/MyDrive/movie_genre_project/final_model")
tokenizer.save_pretrained("/content/drive/MyDrive/movie_genre_project/final_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/movie_genre_project/final_model/tokenizer_config.json',
 '/content/drive/MyDrive/movie_genre_project/final_model/tokenizer.json')

In [ ]:
import torch
import numpy as np

# Load model from Drive (if session restarted)
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

model_path = "/content/drive/MyDrive/movie_genre_project/final_model"

model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizer.from_pretrained(model_path)

model.eval()


# Create label mapping
label_list = list(set(y_train))
label_list.sort()
id2label = {i: label for i, label in enumerate(label_list)}
label2id = {label: i for i, label in enumerate(label_list)}


def predict_genre(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    logits = outputs.logits
    predicted_class_id = torch.argmax(logits, dim=1).item()

    return id2label[predicted_class_id]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

In [ ]:
def predict_genre(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class_id = torch.argmax(outputs.logits, dim=1).item()

    return id2label[predicted_class_id]

In [ ]:
predict_genre("A detective investigates a mysterious murder in a small town")

24

In [ ]:
# Get unique genres in correct sorted order
label_list = sorted(list(set(y_train)))

label2id = {label: i for i, label in enumerate(label_list)}
id2label = {i: label for i, label in enumerate(label_list)}

print(id2label)

{0: 0, 1: 1, 2: 2, 3: 3, 4: 4, 5: 5, 6: 6, 7: 7, 8: 8, 9: 9, 10: 10, 11: 11, 12: 12, 13: 13, 14: 14, 15: 15, 16: 16, 17: 17, 18: 18, 19: 19, 20: 20, 21: 21, 22: 22, 23: 23, 24: 24, 25: 25, 26: 26}


In [ ]:
# Recreate LabelEncoder using full dataset genres
from sklearn.preprocessing import LabelEncoder

label_encoder = LabelEncoder()
label_encoder.fit(df["genre"])

# Build correct mapping
id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in enumerate(label_encoder.classes_)}

print(id2label)

{0: 'action', 1: 'adult', 2: 'adventure', 3: 'animation', 4: 'biography', 5: 'comedy', 6: 'crime', 7: 'documentary', 8: 'drama', 9: 'family', 10: 'fantasy', 11: 'game-show', 12: 'history', 13: 'horror', 14: 'music', 15: 'musical', 16: 'mystery', 17: 'news', 18: 'reality-tv', 19: 'romance', 20: 'sci-fi', 21: 'short', 22: 'sport', 23: 'talk-show', 24: 'thriller', 25: 'war', 26: 'western'}


In [ ]:
def predict_genre(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    predicted_class_id = torch.argmax(outputs.logits, dim=1).item()

    return id2label[predicted_class_id]

In [ ]:
predict_genre("A detective investigates a mysterious murder in a small town")

'thriller'

In [ ]:
!pip install streamlit pyngrok --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 74.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 93.1 MB/s eta 0:00:00


In [ ]:
%%writefile app.py

import streamlit as st
import torch
import torch.nn.functional as F
from transformers import DistilBertForSequenceClassification, DistilBertTokenizer

# Load model from Drive
model_path = "/content/drive/MyDrive/movie_genre_project/final_model"

model = DistilBertForSequenceClassification.from_pretrained(model_path)
tokenizer = DistilBertTokenizer.from_pretrained(model_path)

model.eval()

# You must paste your correct id2label mapping here
id2label = YOUR_ID2LABEL_DICTIONARY_HERE

def predict_genre(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    with torch.no_grad():
        outputs = model(**inputs)

    probabilities = F.softmax(outputs.logits, dim=1)
    confidence, predicted_class_id = torch.max(probabilities, dim=1)

    genre = id2label[predicted_class_id.item()]
    confidence_score = confidence.item() * 100

    return genre, round(confidence_score, 2)

st.title("🎬 Movie Genre Classifier")

user_input = st.text_area("Enter movie plot:")

if st.button("Predict"):
    genre, confidence = predict_genre(user_input)
    st.success(f"Predicted Genre: {genre}")
    st.info(f"Confidence: {confidence}%")

Writing app.py


In [ ]:
!streamlit run app.py & npx localtunnel --port 8501

⠙

⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧Need to install the following packages:
localtunnel@2.0.2
Ok to proceed? (y) 
  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://35.247.180.106:8501

